You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Title: The Future of Artificial Intelligence: Latest Trends and Key Players

Introduction:
- Brief overview of Artificial Intelligence (AI) and its impact on various industries
- Mention the rapid advancements in AI technology and its potential for the future

Key Points:
1. Latest Trends in Artificial Intelligence:
- Machine learning and deep learning algorithms
- Natural language processing and chatbots
- AI-powered automation and robotics
- Ethical considerations in AI development 

2. Key Players in the 

I now can give a great answer

Final Answer:
# The Future of Artificial Intelligence: Latest Trends and Key Players

## Introduction

Artificial Intelligence (AI) has revolutionized various industries with its disruptive technologies and innovative solutions. The impact of AI can be seen in sectors such as healthcare, finance, transportation, and more, where automation and data-driven insights have transformed operations. As we delve into the future of AI, the possibilities seem endless, with rapid advancements in technology paving the way for new opportunities and challenges.

## Latest Trends in Artificial Intelligence

Machine learning and deep learning algorithms have been at the forefront of AI innovation, enabling computers to learn from data and make informed decisions. Natural language processing and chatbots have also gained popularity, making interactions between humans and machines more seamless and efficient. AI-powered automation and robotics have streamlined processes in 

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# The Future of Artificial Intelligence: Latest Trends and Key Players

## Introduction

Artificial Intelligence (AI) has revolutionized various industries with its disruptive technologies and innovative solutions. The impact of AI can be seen in sectors such as healthcare, finance, transportation, and more, where automation and data-driven insights have transformed operations. As we delve into the future of AI, the possibilities seem endless, with rapid advancements in technology paving the way for new opportunities and challenges.

## Latest Trends in Artificial Intelligence

Machine learning and deep learning algorithms have been at the forefront of AI innovation, enabling computers to learn from data and make informed decisions. Natural language processing and chatbots have also gained popularity, making interactions between humans and machines more seamless and efficient. AI-powered automation and robotics have streamlined processes in various industries, increasing productivity and reducing human error. However, ethical considerations in AI development remain a crucial aspect to address, ensuring that AI technologies are used responsibly and ethically.

## Key Players in the AI Industry

In the competitive landscape of the AI industry, key players such as Google DeepMind, IBM Watson, Microsoft Azure AI, Amazon Web Services AI, and OpenAI have been leading the way with their cutting-edge technologies and strategic partnerships. These companies have been instrumental in driving innovation and shaping the future of AI, setting the benchmark for others to follow in terms of research, development, and implementation.

## Noteworthy News in Artificial Intelligence

Recent breakthroughs in AI research have captivated audiences worldwide, showcasing the limitless potential of AI technologies. Industry partnerships and collaborations have also played a significant role in accelerating AI advancements, fostering a culture of collaboration and knowledge sharing. AI applications in healthcare, finance, and transportation have demonstrated the transformative power of AI, improving efficiency, accuracy, and decision-making processes in these critical sectors.

## Conclusion

As we navigate through the ever-evolving landscape of Artificial Intelligence, it is essential to stay informed and engaged with the latest trends, key players, and noteworthy news in the industry. By keeping abreast of AI developments and participating in discussions on AI trends, we can contribute to the growth and success of AI technologies in the future. Let's embrace the opportunities that AI presents and work towards a future where AI enhances our lives and drives innovation across all sectors.

*Stay tuned for more updates on Artificial Intelligence trends, key players in the AI industry, latest news on Artificial Intelligence, and Artificial Intelligence applications by subscribing to our newsletter or following us on social media.*

References:
1. [Forbes article on AI trends in 2021](https://www.forbes.com/ai-trends-2021)
2. [Gartner report on top players in the AI industry](https://www.gartner.com/top-players-ai-industry)
3. [Harvard Business Review analysis on the impact of AI on business](https://hbr.org/impact-of-ai-on-business)

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [15]:
topic = "Data & IA Product Manager"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Data & IA Product Manager.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Content Plan: Data & IA Product Manager

I. Introduction
- Brief overview of the role of a Data & IA Product Manager
- Importance of this role in the tech industry
- Preview of key trends and news in the field

II. Latest Trends, Key Players, and Noteworthy News
A. Trends
1. Utilization of AI and Machine Learning in product management
2. Emphasis on data-driven decision making
3. Adoption of Agile methodologies in product development
B. Key Players
1. Google
2. Amazon
3. Facebook
C. Noteworthy News
1. Recent

I now can give a great answer

Final Answer:
# The Role of Data & IA Product Manager in Tech: Trends and Insights

## Introduction

In the fast-paced world of technology, the role of a Data & IA Product Manager is becoming increasingly vital. These professionals are tasked with harnessing the power of data and artificial intelligence to drive product development and decision-making processes. With the tech industry constantly evolving, it is crucial to stay updated on the latest trends and news in this field to remain competitive and innovative.

## Latest Trends, Key Players, and Noteworthy News

### Trends

The utilization of AI and Machine Learning in product management has revolutionized the way companies approach innovation. By leveraging these technologies, Data & IA Product Managers can analyze vast amounts of data to inform strategic decisions and enhance user experiences. Furthermore, there is a growing emphasis on data-driven decision-making, where insights derived from data 

In [16]:
Markdown(result)

# The Role of Data & IA Product Manager in Tech: Trends and Insights

## Introduction

In the fast-paced world of technology, the role of a Data & IA Product Manager is becoming increasingly vital. These professionals are tasked with harnessing the power of data and artificial intelligence to drive product development and decision-making processes. With the tech industry constantly evolving, it is crucial to stay updated on the latest trends and news in this field to remain competitive and innovative.

## Latest Trends, Key Players, and Noteworthy News

### Trends

The utilization of AI and Machine Learning in product management has revolutionized the way companies approach innovation. By leveraging these technologies, Data & IA Product Managers can analyze vast amounts of data to inform strategic decisions and enhance user experiences. Furthermore, there is a growing emphasis on data-driven decision-making, where insights derived from data analytics drive product development strategies. Agile methodologies are also gaining traction, allowing teams to adapt quickly to market changes and deliver products more efficiently.

### Key Players

Some of the key players in the tech industry leading the way in data and AI product management include Google, Amazon, and Facebook. These tech giants have integrated AI solutions into their products and services, setting the standard for innovation and excellence in the field.

### Noteworthy News

Recent advancements in AI technology have paved the way for exciting possibilities in product management. Success stories of companies implementing data-driven strategies highlight the tangible benefits of leveraging data effectively. However, Data & IA Product Managers also face challenges in the current market, such as ensuring data privacy and security, navigating regulatory requirements, and managing complex datasets.

## Target Audience

The target audience for this blog post includes data analysts and scientists, product managers and developers, tech enthusiasts interested in AI and data management, business professionals looking to enhance their product strategies, and individuals considering a career in data product management. By providing valuable insights and resources, this article aims to engage and inform a diverse range of professionals in the tech industry.

## SEO Keywords

- Data & IA Product Manager
- AI in product management
- Data-driven decision making
- Product development trends
- Key players in tech industry

## Call to Action

As you delve into the world of data and IA product management, remember to stay updated on the latest trends and developments in the field. Share your thoughts and experiences in the comments section below and explore the links to related articles and resources for further reading. Together, we can drive innovation and success in the tech industry.

By following this comprehensive content plan, the blog article on Data & IA Product Manager will engage the target audience with relevant information, while also offering valuable insights and resources for their professional development in the tech industry.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).